# MACURA flies a Drone to a target

A quadrotor learns to fly to a target and hold a stable hover. The fast approach and
deceleration are aggressive: a fixed-horizon rollout (MBPO) over-imagines that phase and
the drone overshoots or destabilizes, while MACURA truncates the imagined rollout when the
model gets uncertain and learns from the trustworthy pieces. We train the four algorithms
(MACURA, MBPO, M2AC, SAC) - shared SAC backbone and ensemble, only the rollout differs -
and compare sample efficiency and stability. Run top to bottom on a GPU runtime.

## **Connect Colab**

In [ ]:
import os, sys, subprocess
from getpass import getpass

REPO   = "silvergjeka22/macura-backflip"
BRANCH = "main"
ROOT   = "/content/macura-backflip"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

## **Setup**

In [ ]:
from src.bootstrap import setup
setup(drive=True, install_deps=True)

In [ ]:
# imports
%run /content/macura-backflip/src/imports.py
set_seed(cfg.SEED)

## **Fresh start**\nWipe old checkpoints, logs, plots and videos on Drive so nothing stale is reloaded.

In [ ]:
import shutil
for d in ("checkpoints", "videos", "plots", "logs"):
    shutil.rmtree(f"{cfg.DRIVE_ROOT}/{d}", ignore_errors=True)
    os.makedirs(f"{cfg.DRIVE_ROOT}/{d}", exist_ok=True)
print("cleared old runs on Drive:", cfg.DRIVE_ROOT)

## **Meet the drone**\nA short video of the quadrotor with random (untrained) thrusts - it wobbles and drifts, and model error compounds fast here. This is what the four algorithms start from.

In [ ]:
# render the untrained drone in an isolated osmesa subprocess (kernel GL is disabled for SB3 safety)
path = record_env_video_subprocess(ROOT, f"{cfg.DRIVE_ROOT}/videos/drone_env.mp4", seconds=6)
if path:
    display(Video(path, embed=True, width=420))

## **Train**\nFour algorithms x seeds, each learning from scratch (no curriculum). A run whose log already exists on Drive is reloaded.

In [ ]:
# optional shared warm-start (off for the drone: cfg.CURRICULUM["enabled"] = False)
init_ckpt = pretrain_shared_policy(cfg.CFG, cfg.DRIVE_ROOT) if cfg.CURRICULUM["enabled"] else None

runs = []
for algo in cfg.ALGORITHMS:
    for seed in cfg.SEEDS:
        log = f"{cfg.DRIVE_ROOT}/logs/{algo}_seed{seed}.json"
        if os.path.exists(log):
            runs.append(json.load(open(log)))
        else:
            runs.append(train_one(algo, cfg.CFG, cfg.DRIVE_ROOT, seed, init_ckpt=init_ckpt))

## **Results**\nSample efficiency (return), target-reached rate, crash rate, final quality, and the MACURA rollout-length/kappa signature.

In [ ]:
plot_sample_efficiency(runs)
plot_success_rate(runs)
plot_failure_rate(runs)
plot_final_quality(runs)
plot_rollout_depth([r for r in runs if r["algo"] == "macura"][0]);

## **Flight demo**\nColab renders videos only if MACURA_RENDER=1. The intended way to watch the drone fly is live on your Mac (see Save).

In [ ]:
videos = record_best_videos(cfg.CFG, cfg.DRIVE_ROOT, device, seeds=cfg.SEEDS, seconds=8) if cfg.RENDER else {}
for name, path in videos.items():
    print(name)
    display(Video(path, embed=True, width=360))

In [ ]:
labels = {name.split("_seed")[0]: path for name, path in videos.items()}
if labels:
    display(Video(stitch_videos_grid(labels, f"{cfg.DRIVE_ROOT}/videos/reel.mp4", cols=2), embed=True, width=520))

## **Save**\nCheckpoints, plots and logs are on Drive under DRIVE_ROOT. Download the `checkpoints/` folder to your Mac and watch the drone fly live:\n\n`mjpython run_live_mac.py --compare runs/checkpoints`

In [ ]:
for r in sorted(runs, key=lambda r: -r["final_eval"]["eval_return"]):
    fe = r["final_eval"]
    print(f'{r["algo"]:7s} seed{r["seed"]}  return {fe["eval_return"]:7.1f}'
          f'  reached {fe["eval_success_rate"]:.2f}  crash {fe["eval_failure_rate"]:.2f}')